# ecg-arrhythmia-mitbih · Quickstart

Three-cell tour of the public Python API. By the end of this notebook you will have:

1. Loaded a pretrained model from the `artifacts/` directory.
2. Classified every detected beat in the bundled `examples/sample_ecg.csv`.
3. Plotted the ECG with N / PVC / PAC labels and a confidence histogram.

Prereqs (from the repo root):

```
pip install -e ".[dev]"            # baseline only
pip install -e ".[dev,deep]"      # also pulls PyTorch for the ResNet backend
pip install matplotlib            # for the plots in this notebook
```

**Medical disclaimer.** This notebook is for research and education only. Don't use it for diagnosis, triage, or anything clinical.

## 1. Load a pretrained model

`ECGClassifier.from_artifacts(...)` sniffs the directory and picks the right backend (baseline LR or ResNet-1D). Pass `prefer=` to force one.

In [ ]:
from pathlib import Path
from ecg_arrhythmia import ECGClassifier

REPO = Path('.').resolve().parent if Path('.').name == 'examples' else Path('.').resolve()
clf = ECGClassifier.from_artifacts(REPO / 'artifacts' / 'baseline', prefer='baseline')
print('Loaded backend:', clf.backend)

## 2. Classify every beat in the bundled sample

`predict_csv` handles loading, resampling, R-peak detection, windowing, and inference. The returned `PredictionResult` is iterable, sliceable, and has helpers for class counts, heart rate, and a JSON-friendly summary.

In [ ]:
result = clf.predict_csv(
    REPO / 'examples' / 'sample_ecg.csv',
    input_fs=360,
    lead='MLII',
)
print(f'Detected {len(result)} beats')
print('Summary:', result.summary())
for beat in result[:5]:
    print(f'  beat {beat.beat_index:>2} t={beat.peak_time_s:5.2f}s {beat.label}  conf={beat.confidence:.3f}')

## 3. Plot the ECG with predicted labels

In [ ]:
import csv
import matplotlib.pyplot as plt
import numpy as np

with (REPO / 'examples' / 'sample_ecg.csv').open() as f:
    reader = csv.reader(f)
    header = next(reader)
    sig = np.array([float(row[0]) for row in reader if row])
fs = 360.0
t = np.arange(len(sig)) / fs

color = {'N': '#3a7d44', 'V': '#d62828', 'a': '#f4a261'}
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5), gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(t, sig, color='#1f3b73', linewidth=0.8)
for beat in result:
    ax1.scatter([beat.peak_time_s], [sig[beat.peak_sample]], color=color[beat.label], s=40, zorder=5)
    ax1.annotate(beat.label, (beat.peak_time_s, sig[beat.peak_sample]),
                 textcoords='offset points', xytext=(0, 8), fontsize=9, ha='center')
ax1.set_xlabel('time (s)')
ax1.set_ylabel('amplitude (mV)')
ax1.set_title(f'Predicted beats ({result.summary()["class_counts"]})')

ax2.hist(result.confidences, bins=20, color='#1f3b73', alpha=0.85)
ax2.set_xlabel('predicted-class confidence')
ax2.set_ylabel('beats')
ax2.set_xlim(0, 1)
fig.tight_layout();

## Where to next

- **Streaming / real-time** &mdash; see `examples/streaming_demo.py` and `ecg_arrhythmia.streaming.StreamingClassifier` for beat-by-beat inference on chunked input.
- **Edge deployment** &mdash; export to ONNX with `python scripts/export_onnx.py baseline ...` (5 KB) or `... resnet ...` (2.2 MB).
- **Train your own** &mdash; `python scripts/fetch_mitbih.py && make train` (or `make train-resnet`).
- **Wearable inputs** &mdash; pass `lead='I'` and the device sampling rate; the pipeline will resample and auto-flip polarity, and emit a domain-shift warning.